# Data Setup and Verification

This notebook helps you download, verify, and explore the required data files for the V1 JAX project.

## Table of Contents
1. [Download Instructions](#1-download-instructions)
2. [File Structure Check](#2-file-structure-check)
3. [Network Data Preview](#3-network-data-preview)
4. [LGN Data Preview](#4-lgn-data-preview)
5. [Stimulus Data Preview](#5-stimulus-data-preview)
6. [Troubleshooting](#6-troubleshooting)

---

## 1. Download Instructions

Download the required data files from:

**Download Link**: [TU Graz Cloud](https://cloud.tugraz.at/index.php/s/JmDakasAHEqsA9J)

After downloading, extract the archive to a directory of your choice.

---

## 2. File Structure Check

First, configure your data path and verify all required files exist.

In [ ]:
import os
from pathlib import Path

# =============================================================================
# CONFIGURE YOUR DATA PATH HERE
# =============================================================================
DATA_DIR = Path("/path/to/your/data")  # MODIFY THIS to your actual data path
GLIF_NETWORK_DIR = DATA_DIR / "GLIF_network"

print(f"Data directory: {DATA_DIR}")
print(f"GLIF network directory: {GLIF_NETWORK_DIR}")

In [ ]:
# Required files checklist
REQUIRED_FILES = {
    "network": [
        GLIF_NETWORK_DIR / "network" / "v1_node_types.csv",
        GLIF_NETWORK_DIR / "network" / "v1_nodes.h5",
        GLIF_NETWORK_DIR / "network_dat.pkl",
        GLIF_NETWORK_DIR / "input_dat.pkl",
    ],
    "lgn": [
        DATA_DIR / "lgn_full_col_cells_3.csv",
        DATA_DIR / "temporal_kernels.pkl",
    ],
    "stimuli": [
        DATA_DIR / "many_small_stimuli.pkl",
        DATA_DIR / "alternate_small_stimuli.pkl",
        DATA_DIR / "EA_LGN.h5",
    ],
}

def check_files():
    """Check if all required files exist."""
    all_ok = True
    total_size = 0
    
    for category, files in REQUIRED_FILES.items():
        print(f"\n[{category.upper()}]")
        for f in files:
            if f.exists():
                size_mb = f.stat().st_size / 1024 / 1024
                total_size += size_mb
                print(f"  OK: {f.name} ({size_mb:.1f} MB)")
            else:
                print(f"  MISSING: {f.name}")
                all_ok = False
    
    print(f"\n{'='*50}")
    if all_ok:
        print(f"All files present! Total size: {total_size:.1f} MB")
    else:
        print("Some files are missing. Please download from the link above.")
    
    return all_ok

files_ok = check_files()

---

## 3. Network Data Preview

Explore the structure of the V1 network data files.

In [ ]:
import pandas as pd
import pickle
import h5py

# Preview v1_node_types.csv
node_types_path = GLIF_NETWORK_DIR / "network" / "v1_node_types.csv"
if node_types_path.exists():
    node_types = pd.read_csv(node_types_path, sep=' ')
    print(f"Number of neuron types: {len(node_types)}")
    print(f"\nColumns: {list(node_types.columns)}")
    print(f"\nFirst 5 rows:")
    display(node_types.head())
else:
    print("v1_node_types.csv not found")

In [ ]:
# Preview v1_nodes.h5 structure
nodes_h5_path = GLIF_NETWORK_DIR / "network" / "v1_nodes.h5"
if nodes_h5_path.exists():
    print("v1_nodes.h5 structure:")
    print("="*50)
    with h5py.File(nodes_h5_path, 'r') as f:
        def print_structure(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
            else:
                print(f"  {name}/")
        f.visititems(print_structure)
else:
    print("v1_nodes.h5 not found")

In [ ]:
# Preview network_dat.pkl
network_dat_path = GLIF_NETWORK_DIR / "network_dat.pkl"
if network_dat_path.exists():
    with open(network_dat_path, 'rb') as f:
        network_dat = pickle.load(f)
    
    print("network_dat.pkl contents:")
    print("="*50)
    for key, value in network_dat.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, dict):
            print(f"  {key}: dict with {len(value)} keys")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("network_dat.pkl not found")

In [ ]:
# Preview input_dat.pkl
input_dat_path = GLIF_NETWORK_DIR / "input_dat.pkl"
if input_dat_path.exists():
    with open(input_dat_path, 'rb') as f:
        input_dat = pickle.load(f)
    
    print("input_dat.pkl contents:")
    print("="*50)
    for key, value in input_dat.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, dict):
            print(f"  {key}: dict with {len(value)} keys")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("input_dat.pkl not found")

---

## 4. LGN Data Preview

Explore the LGN (Lateral Geniculate Nucleus) neuron data.

In [ ]:
import matplotlib.pyplot as plt

# Preview lgn_full_col_cells_3.csv
lgn_csv_path = DATA_DIR / "lgn_full_col_cells_3.csv"
if lgn_csv_path.exists():
    lgn_data = pd.read_csv(lgn_csv_path, delimiter=' ')
    print(f"Number of LGN neurons: {len(lgn_data)}")
    print(f"\nColumns: {list(lgn_data.columns)}")
    print(f"\nFirst 5 rows:")
    display(lgn_data.head())
else:
    print("lgn_full_col_cells_3.csv not found")

In [ ]:
# Visualize LGN neuron positions
if lgn_csv_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Position scatter plot
    axes[0].scatter(lgn_data['x'], lgn_data['y'], s=1, alpha=0.5, c='blue')
    axes[0].set_xlabel('X position')
    axes[0].set_ylabel('Y position')
    axes[0].set_title('LGN Neuron Positions in Visual Field')
    axes[0].set_aspect('equal')
    
    # Spatial size distribution
    axes[1].hist(lgn_data['spatial_size'], bins=30, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Spatial Size')
    axes[1].set_ylabel('Count')
    axes[1].set_title('LGN Spatial Size Distribution')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Preview temporal_kernels.pkl
temporal_kernels_path = DATA_DIR / "temporal_kernels.pkl"
if temporal_kernels_path.exists():
    with open(temporal_kernels_path, 'rb') as f:
        temporal_kernels = pickle.load(f)
    
    print("temporal_kernels.pkl contents:")
    print("="*50)
    for key, value in temporal_kernels.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("temporal_kernels.pkl not found")

In [ ]:
# Visualize sample temporal kernels
if temporal_kernels_path.exists():
    dom_kernels = temporal_kernels['dom_temporal_kernels']
    
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Plot first 10 kernels
    for i in range(min(10, len(dom_kernels))):
        ax.plot(dom_kernels[i], alpha=0.7, label=f'Neuron {i}')
    
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Kernel Value')
    ax.set_title('Sample LGN Temporal Kernels (Dominant Subunit)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---

## 5. Stimulus Data Preview

Explore the stimulus datasets used for training and evaluation.

In [ ]:
# Preview many_small_stimuli.pkl (classification images)
stimuli_path = DATA_DIR / "many_small_stimuli.pkl"
if stimuli_path.exists():
    with open(stimuli_path, 'rb') as f:
        stimuli = pickle.load(f)
    
    print("many_small_stimuli.pkl contents:")
    print("="*50)
    for key, value in stimuli.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, list):
            print(f"  {key}: list with {len(value)} items")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("many_small_stimuli.pkl not found")

In [ ]:
# Visualize sample classification images
if stimuli_path.exists() and 'images' in stimuli:
    images = stimuli['images']
    
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i], cmap='gray')
        ax.axis('off')
    
    plt.suptitle('Sample Classification Stimuli', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print(f"Image shape: {images[0].shape}")
    print(f"Value range: [{images[0].min():.2f}, {images[0].max():.2f}]")

In [ ]:
# Preview alternate_small_stimuli.pkl
alt_stimuli_path = DATA_DIR / "alternate_small_stimuli.pkl"
if alt_stimuli_path.exists():
    with open(alt_stimuli_path, 'rb') as f:
        alt_stimuli = pickle.load(f)
    
    print("alternate_small_stimuli.pkl contents:")
    print("="*50)
    for key, value in alt_stimuli.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, list):
            print(f"  {key}: list with {len(value)} items")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("alternate_small_stimuli.pkl not found")

In [ ]:
# Preview EA_LGN.h5 (evidence accumulation dataset)
ea_path = DATA_DIR / "EA_LGN.h5"
if ea_path.exists():
    print("EA_LGN.h5 structure:")
    print("="*50)
    with h5py.File(ea_path, 'r') as f:
        def print_structure(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
            else:
                print(f"  {name}/")
        f.visititems(print_structure)
else:
    print("EA_LGN.h5 not found")

---

## 6. Troubleshooting

Common issues and solutions:

### Issue: Files not found

**Solution**: Make sure you have:
1. Downloaded the data from [TU Graz Cloud](https://cloud.tugraz.at/index.php/s/JmDakasAHEqsA9J)
2. Extracted the archive
3. Updated the `DATA_DIR` variable in cell 3 to point to your data directory

### Issue: Import errors

**Solution**: Make sure you have installed the package:
```bash
cd /path/to/allen_v1_chen_2022_jax
uv sync
```

### Issue: Memory errors when loading large files

**Solution**: The full network data requires ~4GB RAM. If you have limited memory:
- Use `load_billeh(n_neurons=1000)` to load a smaller subset
- Close other applications to free memory

In [ ]:
# Test loading with v1_jax package
try:
    from v1_jax.data import load_billeh
    print("v1_jax.data imported successfully!")
    
    # Try loading a small subset
    if GLIF_NETWORK_DIR.exists():
        print("\nTesting load_billeh with small subset...")
        input_pop, network, bkg = load_billeh(
            n_neurons=1000,  # Small subset for testing
            data_dir=str(GLIF_NETWORK_DIR),
            seed=42,
        )
        print(f"Loaded {network['n_nodes']} neurons")
        print(f"Loaded {network['n_edges']} synapses")
        print("\nData loading works correctly!")
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure you have installed the package with 'uv sync'")
except Exception as e:
    print(f"Error loading data: {e}")

---

## Summary

In this notebook, you learned:
- Where to download the required data files
- How to verify all files are present
- The structure of network, LGN, and stimulus data
- How to troubleshoot common issues

## Next Steps

Now that your data is set up, proceed to:
- `01_spike_functions.ipynb` - Learn about surrogate gradients (no data required)
- `02_glif3_single_neuron.ipynb` - Explore single neuron dynamics (no data required)